In [1]:
from the_well.data.datasets import TrajectoryData, TrajectoryMetadata
from the_well.data.normalization import ZScoreNormalization
from the_well.data import WellDataset

from the_well.benchmark.metrics.spatial import VRMSE
from typing import Sequence, Dict, Tuple, Optional
from einops import rearrange
from pathlib import Path
import fsspec
import pickle
import torch
import yaml
import sys
import os

from unittest.mock import MagicMock
# Mock torch_harmonics as a proper package with all submodules neuralop touches
mock_th = MagicMock()
mock_th.__path__ = []          # marks it as a package
mock_th.__spec__ = MagicMock() # satisfies importlib checks

sys.modules['torch_harmonics']          = mock_th
sys.modules['torch_harmonics.examples'] = MagicMock()
sys.modules['torch_harmonics.layers']   = MagicMock()
sys.modules['torch_harmonics.grids']    = MagicMock()

from the_well.benchmark.models import UNetClassic, UNetConvNext, FNO, TFNO

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.neural_ops import NeuralOperatorModule

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

/opt/anaconda3/envs/diffusion_env/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# Load Data

In [2]:
well_base_path = f"{PROJECT_ROOT}/data/datasets"

well_dataset_name = 'turbulent_radiative_layer_2D'
#well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'planetswe'

stats_path = f"{well_base_path}/{well_dataset_name}/stats.yaml"

dataset = get_well_dataset(
    path=well_base_path,
    physics=well_dataset_name,
    split='test',
    steps=-1,
    transform = ['log_scalars'],   
    augment = [],
)

with fsspec.open(stats_path, mode="r") as f:
    stats = yaml.safe_load(f)

core_field_names          = dataset.core_field_names
core_constant_field_names = dataset.core_constant_field_names

normalizer = ZScoreNormalization(stats, core_field_names, core_constant_field_names)

# Load Baseline Models

In [ ]:
class CPUForwardWrapper(torch.nn.Module):
    """
    Makes a CPU-resident model usable in an MPS pipeline.
    Input arrives on any device, forward runs on CPU, output
    is returned on the same device as the input.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model.to("cpu")

    def to(self, device, **kwargs):
        # Intercept .to("mps") — keep inner model on CPU always
        self._output_device = torch.device(device) if isinstance(device, str) else device
        return self  # do NOT call super().to() — that would move inner model

    def forward(self, x):
        out = self.model(x.cpu())   # move input to CPU, run, 
        return out.to(x.device)     # return output to original device (mps)

In [3]:
unet_classic  = UNetClassic.from_pretrained(f"polymathic-ai/UNetClassic-{well_dataset_name}")
unet_convnext = UNetConvNext.from_pretrained(f"polymathic-ai/UNetConvNext-{well_dataset_name}")
#fno           = FNO.from_pretrained(f"polymathic-ai/FNO-{well_dataset_name}")
#tfno          = TFNO.from_pretrained(f"polymathic-ai/TFNO-{well_dataset_name}")

from waveletflux.modules.neural_ops import NeuralOperatorModule
fno = NeuralOperatorModule.load_from_checkpoint(f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/fno_turbulent_radiative_layer_2D_v0.0/last_fp32.pt", map_location="cpu")
#.from_pretrained("/Users/gabrieleaccarino/Workspace/waveletflux/experiments/turbulent_radiative_layer_2D/ckpts/fno_turbulent_radiative_layer_2D_v0.0/last_fp32.pt")

baselines = {
    'UNetClassic'   : unet_classic.to(device),
    'UNetConvNext'  : unet_convnext.to(device),
    'FNO'           : fno.to(device),
    #'FNO'           : fno.to(device),
    #'TFNO'          : CPUForwardWrapper(tfno).to(device),
}

/opt/anaconda3/envs/diffusion_env/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.0, which is newer than your current Lightning version: v2.5.5


In [ ]:
"""from torch.utils.data import DataLoader

N_STEPS_INPUT  = 4      # context window fed to the model
MAX_ROLLOUT_STEPS = 16  # how many steps to autoregressively predict
BATCH_SIZE = 1

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

# Quick sanity-check: inspect one raw batch
batch = next(iter(loader))
x, label = get_well_inputs(batch, device=DEVICE)
print("x shape      :", x.shape)   # (B, T_total, H, W, C)
print("label        :", label.shape if label is not None else None)
print("dtype        :", x.dtype)"""

In [4]:
def normalize_x(x: torch.Tensor, normalizer) -> torch.Tensor:
    """Normalize a (B, T, H, W, C) field tensor in-place over the C dimension."""
    # ZScoreNormalization.normalize_flattened expects (B, T, C) or (B, C) depending on impl;
    # reshape to match what the Well normalizer expects.
    B, T, *spatial, C = x.shape
    flat = x.reshape(B * T, *spatial, C)          # merge batch & time → treat as batch
    flat = normalizer.normalize_flattened(flat, "variable")
    return flat.reshape(B, T, *spatial, C)

def denormalize_x(x: torch.Tensor, normalizer) -> torch.Tensor:
    """Inverse of normalize_x."""
    B, T, *spatial, C = x.shape
    flat = x.reshape(B * T, *spatial, C)
    flat = normalizer.denormalize_flattened(flat, "variable")
    return flat.reshape(B, T, *spatial, C)

In [10]:
@torch.inference_mode()
def rollout_normalized(
    model,
    x_full: torch.Tensor,        # (T_total, H, W, C) — raw physical space
    normalizer,
    n_steps_input:     int,
    rollout_steps:     int,       # explicit arg — no magic T_total-n_steps_input inside
    device: torch.device = 'cpu',
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Returns
    -------
    y_pred_norm : (rollout_steps, H, W, C)  normalized  ← use for VRMSE
    y_ref_norm  : (rollout_steps, H, W, C)  normalized  ← use for VRMSE
    y_pred_phys : (rollout_steps, H, W, C)  physical    ← use for plots
    y_ref_phys  : (rollout_steps, H, W, C)  physical    ← use for plots
    """
    x_full = x_full.to(device)
    _, H, W, C = x_full.shape

    # Normalize once up front
    x_norm_full = normalize_x(x_full.clone(), normalizer)   # (T_total, H, W, C)

    # Ground truth window: frames [n_steps_input, n_steps_input + rollout_steps)
    y_ref_norm = x_norm_full[n_steps_input : n_steps_input + rollout_steps]
    y_ref_phys = x_full     [n_steps_input : n_steps_input + rollout_steps]

    window_norm = x_norm_full[:n_steps_input].clone()   # (n_in, H, W, C)
    y_preds_norm = []

    for step in range(rollout_steps):
        inp = rearrange(window_norm, 'T H W C -> 1 (T C) H W')
        y_pred_norm = model(inp)                            # (1, T_out*C, H, W)
        y_pred_norm = rearrange(y_pred_norm, '1 (T C) H W -> T H W C', C=C)
        y_preds_norm.append(y_pred_norm)

        if step < rollout_steps - 1:
            window_norm = torch.cat([window_norm[1:], y_pred_norm], dim=0)

    y_pred_norm_out = torch.cat(y_preds_norm, dim=0)       # (rollout_steps, H, W, C)
    y_pred_phys = denormalize_x(y_pred_norm_out.clone(), normalizer)

    return (
        y_pred_norm_out.cpu(),
        y_ref_norm.cpu(),
        y_pred_phys.cpu(),
        y_ref_phys.cpu(),
    )

In [12]:
N_TRAJ        = len(dataset.metadata.n_steps_per_trajectory)                # number of trajectories in the test set for this PDE 
N_STEPS_INPUT = 4                                                           # N_STEPS_INPUT sliding-window width (must match training)
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0] - N_STEPS_INPUT  # autoregressive steps after the seed window

baseline_results = {name: [] for name in baselines}
truth_list       = []   # one (T_total, H, W, C) per trajectory

for traj_idx in range(N_TRAJ):
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item = dataset[traj_idx]
    x_full, constants = get_well_inputs(item, device=device)   # (T_total, H, W, C)
    truth_list.append(x_full.cpu())

    # Run rollout
    for model_name, model in baselines.items():
        print(f"    Running rollout for {model_name}...")
        model.eval()
        y_pred_norm, y_ref_norm, y_pred_phys, y_ref_phys = rollout_normalized(
            model, x_full, normalizer,
            n_steps_input=N_STEPS_INPUT,
            rollout_steps=ROLLOUT_STEPS,
            device=device,
        )
        baseline_results[model_name].append(y_pred_phys)

# Stack to (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
for name in baselines:
    baseline_results[name] = torch.stack(baseline_results[name], dim=0)

truth_traj = torch.stack(truth_list, dim=0)   # (N_TRAJ, T_total, H, W, C)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Running rollout for UNetClassic...
    Running rollout for UNetConvNext...
    Running rollout for FNO...

━━━━━━━

In [ ]:
%matplotlib inline

from matplotlib import pyplot as plt

traj_idx = 5
step = 16
field_idx = 0 

yhat = results['FNO'][traj_idx][step, ..., field_idx].cpu()
y = ground_truth[traj_idx][step+N_STEPS_INPUT, ..., field_idx].cpu()

fig = plt.figure(figsize=(12, 5))

vmin = y.min()
vmax = y.max()

ax0 = fig.add_subplot(1, 2, 1)
truth = ax0.imshow(y, cmap = 'RdBu_r', vmin=vmin, vmax=vmax)
ax0.set_title("Reference")
cbar = fig.colorbar(truth, ax=ax0, fraction=0.025, pad=0.04)

ax1 = fig.add_subplot(1, 2, 2)
pred = ax1.imshow(yhat, cmap = 'RdBu_r', vmin=vmin, vmax=vmax)
#yhat = ax1.imshow(y_denorm[step, ..., field_idx], cmap = 'RdBu_r', vmin=vmin, vmax=vmax)
ax1.set_title("Prediction")
cbar = fig.colorbar(pred, ax=ax1, fraction=0.025, pad=0.04)

plt.show()

In [ ]:
from the_well.benchmark.metrics import VRMSE
from matplotlib import pyplot as plt

# VRMSE must be computed in normalized space (per discussion #28)
# Add batch dim: VRMSE.eval expects (B, T, H, W, C)
vrmse_per_step = VRMSE.eval(
    y_pred_norm.unsqueeze(0),
    y_ref_norm.unsqueeze(0),
    dataset.metadata,
)
print("VRMSE per step:", vrmse_per_step)
print("Mean VRMSE    :", vrmse_per_step.mean().item())

# Amplitude check now in physical space — should no longer collapse
steps = range(y_pred_phys.shape[0])
plt.figure(figsize=(10, 4))
plt.plot([y_ref_phys [t].max().item() for t in steps], label='ref max',  color='blue')
plt.plot([y_pred_phys[t].max().item() for t in steps], label='pred max', color='blue', ls='--')
plt.plot([y_ref_phys [t].min().item() for t in steps], label='ref min',  color='red')
plt.plot([y_pred_phys[t].min().item() for t in steps], label='pred min', color='red', ls='--')
plt.legend(); plt.xlabel('rollout step')
plt.title('Amplitude — should track reference better now')
plt.tight_layout(); plt.show()

In [ ]:
%matplotlib inline

from matplotlib import pyplot as plt

#y_denorm = denormalize_x(y_pred, normalizer)

step = 0
field_idx = 0 

fig = plt.figure(figsize=(12, 5))

vmin = y_ref_phys[step, ..., field_idx].min()
vmax = y_ref_phys[step, ..., field_idx].max()

ax0 = fig.add_subplot(1, 2, 1)
y = ax0.imshow(y_ref_phys[step, ..., field_idx], cmap = 'RdBu_r', vmin=vmin, vmax=vmax)
ax0.set_title("Reference")
cbar = fig.colorbar(y, ax=ax0, fraction=0.025, pad=0.04)

ax1 = fig.add_subplot(1, 2, 2)
yhat = ax1.imshow(y_pred_phys[step, ..., field_idx], cmap = 'RdBu_r', vmin=vmin, vmax=vmax)
#yhat = ax1.imshow(y_denorm[step, ..., field_idx], cmap = 'RdBu_r', vmin=vmin, vmax=vmax)
ax1.set_title("Prediction")
cbar = fig.colorbar(yhat, ax=ax1, fraction=0.025, pad=0.04)

plt.show()

In [ ]:
@torch.inference_mode()
def rollout(
    model,
    x_full: torch.Tensor,       # (B, T_total, H, W, C)  — raw, physical space
    normalizer,
    n_steps_input:    int,
    max_rollout_steps: int,
    device: torch.device = 'cpu',
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Autoregressively roll out the model in physical space.

    Returns
    -------
    y_pred : (B, rollout_steps, H, W, C)  — physical space
    y_ref  : (B, rollout_steps, H, W, C)  — physical space
    """
    N, H, W, C = x_full.shape
    #print(f"x_full shape: {x_full.shape}")
    x_full = x_full.to(device)
    #rollout_steps = min(x_full.shape[0] - n_steps_input, max_rollout_steps)
    
    if max_rollout_steps < 0:
        rollout_steps = N - n_steps_input
    else:
        rollout_steps = min(N - n_steps_input, max_rollout_steps)

    y_ref  = x_full[n_steps_input : n_steps_input + rollout_steps]  # physical

    # Sliding context window — kept in physical space, normalised per step
    window = x_full[:n_steps_input].clone()   # (B, N_in, H, W, C)

    y_preds = []
    for step in range(rollout_steps):
        #print(f'window  : {window.shape}')
        # --- normalise current window -----------------------------------------
        window_norm = normalize_x(window.clone(), normalizer)               # (B, T, H, W, C)
        #print(f'window_norm : {window_norm.shape}')
        window_norm = rearrange(window_norm, 'T H W C -> 1 (T C) H W')    # (B, T, C, H, W) if model expects channel-first

        # --- forward pass (model receives normalised inputs) ------------------
        y_pred_norm = model(window_norm)                                    # (B, 1, H, W, C)  or (B,H,W,C)
        #print(f'y_pred_norm : {y_pred_norm.shape}')

        # Ensure time dimension is present
        #if y_pred_norm.ndim == x_full.ndim - 1:
        #    y_pred_norm = y_pred_norm.unsqueeze(1)
        y_pred_norm = rearrange(y_pred_norm, '1 (T C) H W -> T H W C', C=C)  # reshape back to (T, H, W, C)

        # --- denormalise prediction -------------------------------------------
        y_pred_phys = denormalize_x(y_pred_norm.clone(), normalizer)  # (B,1,H,W,C)
        #print(f'y_pred_phys : {y_pred_phys.shape}')

        y_preds.append(y_pred_phys)

        #window_norm_2 = normalize_x(window[1:, ...], normalizer)

        # --- slide window forward (autoregressive) ----------------------------
        if step < rollout_steps - 1:
            window = torch.cat([window[1:, ...], y_pred_phys], dim=0)

    y_pred_out = torch.cat(y_preds, dim=0)  # (B, rollout_steps, H, W, C)
    return y_pred_out.detach().cpu(), y_ref.detach().cpu()

In [ ]:
# ── test on one batch ───────────────────────────────────────────────────────
batch    = dataset[0]
x, label = get_well_inputs(batch, device=device)
print("x shape      :", x.shape)   # (B, T_total, H, W, C)

y_pred, y_ref = rollout(
    model, x, normalizer,
    n_steps_input=4,
    max_rollout_steps=-1,
    device=device,
)
print("y_pred shape:", y_pred.shape)   # (B, rollout_steps, H, W, C)
print("y_ref  shape:", y_ref.shape)

In [ ]:
print(y_pred.min(), y_pred.max())
print(y_ref.min(), y_ref.max())
y_pred_norm = normalize_x(y_pred, normalizer)
y_ref_norm  = normalize_x(y_ref, normalizer)
print(y_pred_norm.min(), y_pred_norm.max())
print(y_ref_norm.min(), y_ref_norm.max())


In [ ]:
VRMSE_metric = VRMSE().eval(y_ref_norm, y_pred_norm, meta=dataset.metadata)

In [ ]:
VRMSE_metric[6:12, :].mean()

In [ ]:
%matplotlib inline

from matplotlib import pyplot as plt

#y_denorm = denormalize_x(y_pred, normalizer)

step = 0
field_idx = 0 

fig = plt.figure(figsize=(12, 5))

ax0 = fig.add_subplot(1, 2, 1)
y = ax0.imshow(y_ref_norm[step, ..., field_idx], cmap = 'RdBu_r')
ax0.set_title("Reference")
cbar = fig.colorbar(y, ax=ax0, fraction=0.025, pad=0.04)

ax1 = fig.add_subplot(1, 2, 2)
yhat = ax1.imshow(y_pred_norm[step, ..., field_idx], cmap = 'RdBu_r') #, vmin=vmin, vmax=vmax)
#yhat = ax1.imshow(y_denorm[step, ..., field_idx], cmap = 'RdBu_r', vmin=vmin, vmax=vmax)
ax1.set_title("Prediction")
cbar = fig.colorbar(yhat, ax=ax1, fraction=0.025, pad=0.04)

plt.show()

In [ ]:
VRMSE_metric[13:30, :].mean()

In [ ]:
#   class CPUForwardWrapper(torch.nn.Module):
#       """
#       Makes a CPU-resident model usable in an MPS pipeline.
#       Input arrives on any device, forward runs on CPU, output
#       is returned on the same device as the input.
#       """
#       def __init__(self, model):
#           super().__init__()
#           self.model = model.to("cpu")
#   
#       def to(self, device, **kwargs):
#           # Intercept .to("mps") — keep inner model on CPU always
#           self._output_device = torch.device(device) if isinstance(device, str) else device
#           return self  # do NOT call super().to() — that would move inner model
#   
#       def forward(self, x):
#           out = self.model(x.cpu())   # move input to CPU, run, 
#           return out.to(x.device)     # return output to original device (mps)
#       
#   SPECTRAL_MODELS = {"FNO", "TFNO"}
#   dname = dataset.metadata.dataset_name
#   device = torch.device("mps")
#   
#   baselines = {}
#   for name, cls in [("FNO",          FNO),
#                     ("TFNO",         TFNO),
#                     ("UNetClassic",  UNetClassic),
#                     ("UNetConvNext", UNetConvNext)]:
#       model = cls.from_pretrained(f"polymathic-ai/{name}-{dname}").eval()
#       if name in SPECTRAL_MODELS:
#           model = CPUForwardWrapper(model)
#       else:
#           model = model.to(device)
#       baselines[name] = model

In [ ]:
"""N_TRAJ        = len(dataset.metadata.n_steps_per_trajectory)   # which trajectories to evaluate 
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0]     # autoregressive steps after the seed window
N_STEPS_INPUT = 4                                              # N_STEPS_INPUT sliding-window width (must match training)

results = {name: [] for name in list(baselines.keys())}
ground_truth = []

IS_RESIDUAL = {
    "FNO":  True,
    "TFNO": True,
    "UNetClassic": False,
    "UNetConvNext": False,
}

# Safe range based on your diagnostic: seed window was [-1.49, 2.50]
# 3.0 gives comfortable headroom without being too tight
NORM_CLAMP = {
    "FNO":          None, #(-3.5, 3.5),
    "TFNO":         None, #(-3.5, 3.5),
    "UNetClassic":  None,           # no clamping for UNets
    "UNetConvNext": None,
}


for traj_idx in range(N_TRAJ):
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item      = dataset[traj_idx]
    x_full, _ = get_well_inputs(item, device=device)
    ground_truth.append(x_full)                           # list of (T, H, W, C)
    H, W, C = x_full.shape[1:]  # (T, H, W, C)

    # ════════════════════════════════════════════════════════════════════════
    # BASELINES — single deterministic prediction per step
    # ════════════════════════════════════════════════════════════════════════
    for model_name, model in baselines.items():
        print(f"\n  [{model_name}]")
        model.eval()
        model.to(device)

        seed_raw    = x_full[:N_STEPS_INPUT].unsqueeze(0).to(device)
        window_norm = normalizer.normalize_flattened(seed_raw, mode='variable')             # (1, N_STEPS_INPUT, H, W, C)
        bl_preds = torch.zeros(ROLLOUT_STEPS, H, W, C)
        clamp_range = NORM_CLAMP[model_name]

        with torch.no_grad():
            for step in range(ROLLOUT_STEPS):
                gt_idx = N_STEPS_INPUT + step
                if gt_idx >= len(x_full):
                    break

                x_in      = rearrange(window_norm, 'B T H W C -> B (T C) H W')
                yhat      = model(x_in)                                                     # (B, C_out, H, W)
                
                if IS_RESIDUAL[model_name]:
                    # FNO predicts Δx — add back the last frame in normalized space
                    last_hw = rearrange(window_norm[:, -1:], 'B T H W C -> B (T C) H W')
                    yhat    = last_hw + yhat                     # (B, C, H, W)
                
                yhat_norm = rearrange(yhat, 'B C H W -> B 1 H W C')

                # Clamp in normalized space to prevent spectral mode blow-up (FNO only)
                if clamp_range is not None:
                    yhat_norm = yhat_norm.clamp(*clamp_range)

                pred_phys = normalizer.denormalize_flattened(yhat_norm, mode='variable')    # (1, 1, H, W, C)

                #bl_preds[step] = pred_phys.cpu().squeeze(0).squeeze(0)
                bl_preds[step] = pred_phys.squeeze(0).squeeze(0)

                #next_frame_norm = normalizer.normalize_flattened(pred_phys, mode='variable')
                #window_norm = torch.cat([window_norm[:, 1:], next_frame_norm.detach()], dim=1)

                window_norm = torch.cat(
                    [window_norm[:, 1:], yhat_norm.detach()], dim=1)

                print(f"    step {step+1:>4d}/{ROLLOUT_STEPS-N_STEPS_INPUT}", end='\r')

        print()
        results[model_name].append(bl_preds)                            # list of (T, H, W, C)"""

In [ ]:
dset_norm = normalizer

def normalize(batch_dict=None, direct_tensor=None):
        if dset_norm:
            if batch_dict is not None:
                batch_dict["input_fields"] = dset_norm.normalize_flattened(
                    batch_dict["input_fields"], "variable"
                )
                #if "constant_fields" in batch_dict:
                #    batch_dict["constant_fields"] = dset_norm.normalize_flattened(
                #        batch_dict["constant_fields"], "constant"
                #    )
            if direct_tensor is not None:
                    direct_tensor = dset_norm.normalize_flattened(
                        direct_tensor, "variable"
                    )
        return batch_dict, direct_tensor

def denormalize(batch_dict=None, direct_tensor=None):
        if dset_norm:
            if batch_dict is not None:
                batch_dict["input_fields"] = dset_norm.denormalize_flattened(
                    batch_dict["input_fields"], "variable"
                )
                #if "constant_fields" in batch_dict:
                #    batch_dict["constant_fields"] = (
                #        dset_norm.denormalize_flattened(
                #            batch_dict["constant_fields"], "constant"
                #        )
                #    )
            if direct_tensor is not None:
                direct_tensor = dset_norm.denormalize_flattened(
                    direct_tensor, "variable"
                )

        return batch_dict, direct_tensor

In [ ]:
#item      = dataset[0]
#x_full, const = get_well_inputs(item)

In [ ]:
from the_well.data.data_formatter import (
    DefaultChannelsFirstFormatter,
    DefaultChannelsLastFormatter,
)
is_delta = False

formatter = DefaultChannelsFirstFormatter(dataset.metadata)

N_TRAJ        = 1 #len(dataset.metadata.n_steps_per_trajectory)   # which trajectories to evaluate 
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0]     # autoregressive steps after the seed window
N_STEPS_INPUT = 4                                              # N_STEPS_INPUT sliding-window width (must match training)

results = {name: [] for name in list(baselines.keys())}
ground_truth = []

for traj_idx in range(N_TRAJ):
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item      = dataset[traj_idx]
    x_full, constants = get_well_inputs(item, device=device)
    ground_truth.append(x_full)                           # list of (T, H, W, C)
    H, W, C = x_full.shape[1:]  # (T, H, W, C)

    # ════════════════════════════════════════════════════════════════════════
    # BASELINES — single deterministic prediction per step
    # ════════════════════════════════════════════════════════════════════════
    for model_name, model in baselines.items():
        print(f"\n  [{model_name}]")
        model.eval()
        model.to(device)

        #seed_raw    = x_full[:N_STEPS_INPUT].unsqueeze(0).to(device)
        #window_norm = normalizer.normalize_flattened(seed_raw, mode='variable')             # (1, N_STEPS_INPUT, H, W, C)
        
        # seed window in PHYSICAL space — re-normalize each step (mirrors official trainer)
        window_phys = x_full[:N_STEPS_INPUT].unsqueeze(0).to(device)       # (1,N_IN,H,W,C)

        bl_preds = torch.zeros(ROLLOUT_STEPS, H, W, C)


        with torch.no_grad():
            for step in range(ROLLOUT_STEPS):
                # 1. Normalize window fresh from physical space
                #window_norm, _ = normalizer.normalize({"input_fields": window_phys, "constant_fields": constants})
                window_norm, _ = normalize(batch_dict={"input_fields": window_phys})
                # 2. Use the ACTUAL formatter, not manual rearrange
                #inputs, _ = formatter.process_input({'input_fields': window_norm['input_fields'], 'constant_fields': constants.expand(window_norm['input_fields'].shape)})
                inputs, _ = formatter.process_input({'input_fields': window_norm['input_fields'], 'output_fields': torch.zeros_like(window_norm['input_fields'])})
                inputs = [x.to(device) for x in inputs]

                #print(inputs[0].min(), inputs[0].max())

                # 3. Forward
                yhat = model(inputs[0])

                # 4. Formatter output postprocessing
                yhat = formatter.process_output_channel_last(yhat)   # → (B,1,H,W,C)

                # 5. Denormalize
                _, yhat_phys = denormalize(None, yhat)

                # 6. Delta correction if needed
                #if is_delta:
                #    yhat_phys = window_phys[:, -1:] + yhat_phys
                yhat_phys = yhat_phys.unsqueeze(1)

                #print(window_phys[:, 1:].shape, yhat_phys.shape)

                # 7. Slide window in physical space
                window_phys = torch.cat([window_phys[:, 1:], yhat_phys], dim=1)

                print(f"    step {step+1:>4d}/{ROLLOUT_STEPS-N_STEPS_INPUT}", end='\r')

        print()
        results[model_name].append(bl_preds)                            # list of (T, H, W, C)

In [ ]:
results['UNetConvNext'][0].shape, ground_truth[0].shape

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

fig = plt.figure(figsize=(12, 6))

ax0 = fig.add_subplot(1, 2, 1)
ax0.imshow(results['FNO'][0][0, ..., 0].cpu(), label='FNO', cmap = 'RdBu_r')

ax1 = fig.add_subplot(1, 2, 2)
ax1.imshow(ground_truth[0][0, ..., 0].cpu(), label='Ground Truth', cmap = 'RdBu_r')

plt.show()

# Save baselines to binary

In [15]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    results[model_name] = torch.stack(results[model_name], dim = 0).cpu()     # (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
    save_path = save_dir / f"{model_name}_{well_dataset_name}_rollout_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(results[model_name], f)

# Save Ground-truth to binary

In [16]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    gt = torch.stack(ground_truth, dim = 0).cpu()     # (N_TRAJ, N_STEPS, H, W, C)
    save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(gt, f)